In [1]:
# N-1 — Environment. Afterwards: Runtime → Restart session, then run N-2 onward.
!apt-get -qq install -y git-lfs > /dev/null
import subprocess
from pathlib import Path
ROOT = Path("/content/Project-phase-2")
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/CMU-18734-17731-2026-Fall/Project-phase-2.git", str(ROOT)], check=True)
subprocess.run(["git", "lfs", "install"], check=True)
subprocess.run(["git", "lfs", "pull"], cwd=ROOT, check=True)
%pip install -q "transformers==4.56.2" "peft==0.13.2" "accelerate==1.10.1" "datasets==4.1.1"
print("Done. Restart the session now.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 193.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 168.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requ

In [2]:
# N-2 — Imports and constants. Nothing inherited from earlier runtimes.
import os, sys, gc, json, random, shutil, hashlib, subprocess
os.environ.update(USE_TF="0", USE_FLAX="0", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, roc_curve
import transformers, peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
assert transformers.__version__ == "4.56.2" and peft.__version__ == "0.13.2"
assert torch.cuda.is_available(), "GPU runtime required"

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/Project-phase-2")
DRIVE = Path("/content/drive/MyDrive/MIA_phase2")
OLD = DRIVE / "cache"
MODEL_NAME, MAX_LENGTH = "gpt2_3_lora32_adamw_b8_lr2", 512
PHASES = ("train", "val", "final")                    # "train" = the dev target's evaluation texts
PAIRS = [("inA", "inB"), ("p2A", "p2B"), ("p3A", "p3B"), ("p4A", "p4B")]
NEW_PAIRS = [("p5A", "p5B"), ("p6A", "p6B")]
NEW_PAIR_NO = {("p5A", "p5B"): 5, ("p6A", "p6B"): 6}
REF_NAMES = [n for p in PAIRS for n in p]
KAPPA = 8
device, model_dtype = torch.device("cuda"), torch.float16
env = os.environ.copy()
assert ROOT.exists() and DRIVE.exists()
print("OK")

Mounted at /content/drive
OK


In [3]:
# N-3 — Texts in original order, labels, tokenizer, membership, global evaluation groups.
def read_json(p):
    with Path(p).open("r", encoding="utf-8") as f:
        return json.load(f)

TEXTS = {ph: read_json(ROOT / "data" / ph / "test.json") for ph in PHASES}
LABELS = np.asarray(read_json(ROOT / "data" / "train" / "test_label.json"), dtype=np.int64)
assert all(len(TEXTS[ph]) == 2000 for ph in PHASES) and LABELS.sum() == 1000
DEV_ORDER_SHA = hashlib.sha256("\x1e".join(TEXTS["train"]).encode()).hexdigest()

ADAPTER_DIR = ROOT / "models" / "train" / MODEL_NAME
adapter_config = read_json(ADAPTER_DIR / "adapter_config.json")
BASE_MODEL_NAME = adapter_config["base_model_name_or_path"]
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
SHADOW_TRAIN_SIZE = len(read_json(ROOT / "data" / "train" / "train_finetune.json"))
assert SHADOW_TRAIN_SIZE == 10000

norm = lambda t: " ".join(t.split())
def prefix_keys(texts):
    keys = []
    for s in range(0, len(texts), 256):
        for ids in tokenizer(texts[s:s + 256], add_special_tokens=False, truncation=True,
                             max_length=MAX_LENGTH, padding=False)["input_ids"]:
            keys.append(hashlib.sha256(np.asarray(ids, dtype="<i4").tobytes()).hexdigest())
    return keys
EVAL_NORM = {ph: [norm(t) for t in TEXTS[ph]] for ph in PHASES}
EVAL_PREF = {ph: prefix_keys(TEXTS[ph]) for ph in PHASES}

def membership(train_texts):
    ns = {norm(t) for t in train_texts}; ps = set(prefix_keys(list(train_texts)))
    return {ph: np.array([(a in ns) or (b in ps) for a, b in zip(EVAL_NORM[ph], EVAL_PREF[ph])]) for ph in PHASES}
def shadow_texts(n):
    return [r if isinstance(r, str) else r["text"] for r in read_json(DRIVE / "data" / f"shadow_{n}" / "train_finetune.json")]

MEM = {n: membership(shadow_texts(n)) for n in REF_NAMES}
for a, b in PAIRS[1:]:
    for ph in PHASES:
        assert (MEM[a][ph] ^ MEM[b][ph]).all(), (a, b, ph)
assert int((MEM["inA"]["train"] & MEM["inB"]["train"]).sum()) == 7

# Global groups over dev+val+final (same item order and keys as the original construction).
EVAL_ITEMS = [(ph, i) for ph in PHASES for i in range(2000)]
parent = list(range(len(EVAL_ITEMS)))
def _find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
first = {}
for k, (ph, i) in enumerate(EVAL_ITEMS):
    for key in (("n", EVAL_NORM[ph][i]), ("p", EVAL_PREF[ph][i])):
        if key in first: parent[_find(k)] = _find(first[key])
        else: first[key] = k
GROUPS = {}
for k in range(len(EVAL_ITEMS)):
    GROUPS.setdefault(_find(k), []).append(k)
EVAL_NORM_SET = {EVAL_NORM[ph][i] for ph, i in EVAL_ITEMS}
EVAL_PREF_SET = {EVAL_PREF[ph][i] for ph, i in EVAL_ITEMS}

def assign_sides(pair_no):
    rng = np.random.default_rng(7000 + pair_no)
    side_items, side_n = ([], []), [0, 0]
    for g in rng.permutation(sorted(GROUPS)):
        g = int(g)
        s = int(rng.integers(2)) if side_n[0] == side_n[1] else int(side_n[1] < side_n[0])
        side_items[s].extend(GROUPS[g]); side_n[s] += len(GROUPS[g])
    return side_items, side_n

# Reproduction check: the assignment rule must regenerate pairs 2–4 exactly from their saved datasets.
for k in (2, 3, 4):
    side_items, _ = assign_sides(k)
    flag = np.zeros(len(EVAL_ITEMS), bool); flag[side_items[1]] = True
    got = np.concatenate([MEM[f"p{k}B"][ph] for ph in PHASES])
    assert np.array_equal(flag, got), f"group assignment does not reproduce pair {k}"
print(f"{len(EVAL_ITEMS)} eval items in {len(GROUPS)} groups; assignment rule reproduces pairs 2–4.")

6000 eval items in 5981 groups; assignment rule reproduces pairs 2–4.


In [4]:
# N-4 — Cached losses, alignment checks, frozen LR4 reproduction, bootstrap indices.
def split_npz(path, key="losses"):
    z = np.load(path)
    return [a.astype(np.float64) for a in np.split(z[key], np.cumsum(z["lengths"])[:-1])]

TGT = {"train": split_npz(OLD / "train_base_target_losses.npz", "target_losses"),
       "val":   split_npz(OLD / "val_ref_train_losses.npz"),
       "final": split_npz(OLD / "final_train_tgt_losses.npz")}
REF = {n: {ph: split_npz(OLD / f"{ph}_shadow_{n}_losses.npz") for ph in PHASES} for n in REF_NAMES}
for ph in PHASES:
    exp_len = np.array([len(ids) - 1 for ids in tokenizer(TEXTS[ph], truncation=True, max_length=MAX_LENGTH)["input_ids"]])
    assert np.array_equal(exp_len, [a.size for a in TGT[ph]]), ph
    for n in REF_NAMES:
        assert all(r.shape == t.shape for r, t in zip(REF[n][ph], TGT[ph])), (n, ph)

fixed = [split_npz(OLD / "train_shadow_1001_losses.npz", "shadow_losses"), split_npz(OLD / "train_shadow_1002_losses.npz"),
         split_npz(OLD / "train_ref_val_losses.npz"), split_npz(OLD / "train_ref_final_losses.npz")]
FIX4 = np.stack([np.array([a.mean() for a in f]) for f in fixed])   # shadow1001, shadow1002, val tgt, final tgt
DEV_MT = np.array([a.mean() for a in TGT["train"]])
DEV_L = np.array([a.size for a in TGT["train"]], dtype=np.float64)
mean_loss = lambda n: np.array([a.mean() for a in REF[n]["train"]])

def tpr_strict(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    return float(np.max(tpr[fpr < 0.01]))
def strict_fast(s, y):
    neg, pos = s[y == 0], s[y == 1]
    return np.mean(pos > np.sort(neg)[::-1][int(np.ceil(0.01 * neg.size)) - 1])

def lr_frozen(mt, L, P, Pin, kappa=KAPPA):
    """Frozen LR4: unequal-variance Gaussian LR on √T·mean loss, κ-shrinkage to pooled IN/OUT variances, ×(1+1/n)."""
    sq = np.sqrt(L); U, u = P * sq, mt * sq
    fin = np.isfinite(P); m_in, m_out = Pin & fin, ~Pin & fin
    n_in, n_out = m_in.sum(0), m_out.sum(0)
    assert (n_in > 0).all() and (n_out > 0).all()
    Uz = np.where(fin, U, 0.0)
    mu_in = np.where(m_in, Uz, 0.0).sum(0) / n_in
    mu_out = np.where(m_out, Uz, 0.0).sum(0) / n_out
    rss_in = np.where(m_in, (Uz - mu_in) ** 2, 0.0).sum(0)
    rss_out = np.where(m_out, (Uz - mu_out) ** 2, 0.0).sum(0)
    df_in, df_out = np.maximum(n_in - 1, 0), np.maximum(n_out - 1, 0)
    pool_out = rss_out.sum() / df_out.sum()
    pool_in = rss_in.sum() / df_in.sum() if df_in.sum() > 0 else pool_out
    s_in = (rss_in + kappa * pool_in) / (df_in + kappa) * (1 + 1 / n_in)
    s_out = (rss_out + kappa * pool_out) / (df_out + kappa) * (1 + 1 / n_out)
    A = (-(u - mu_in) ** 2 / (2 * s_in) - 0.5 * np.log(s_in)) - (-(u - mu_out) ** 2 / (2 * s_out) - 0.5 * np.log(s_out))
    assert np.isfinite(A).all()
    return A

S4 = np.load(OLD / "staged4_dev_scores_4pairs.npz")
LR4_DEV = S4["LR (all OUT, κ=8 fixed)"].astype(np.float64)
P_lr4 = np.vstack([np.stack([mean_loss(n) for n in REF_NAMES]), FIX4])
PIN_lr4 = np.vstack([np.stack([MEM[n]["train"] for n in REF_NAMES]), np.zeros(FIX4.shape, bool)])
assert np.allclose(lr_frozen(DEV_MT, DEV_L, P_lr4, PIN_lr4), LR4_DEV, rtol=0, atol=1e-6), "LR4 not reproduced"
assert np.isclose(tpr_strict(LABELS, LR4_DEV), read_json(DRIVE / "frozen_LR4_k8" / "settings.json")["dev"]["LR"])

# Same STAGED-5 mock numbers as before (pairs 1–2 refs + shadows 1001/1002), as a second reproduction check.
R0 = ["inA", "inB", "p2A", "p2B"]
P0 = np.vstack([np.stack([mean_loss(n) for n in R0]), FIX4[:2]])
PIN0 = np.vstack([np.stack([MEM[n]["train"] for n in R0]), np.zeros((2, 2000), bool)])
for t, exp in (("p3A", 0.8483), ("p3B", 0.8084), ("p4A", 0.7450), ("p4B", 0.6586)):
    got = tpr_strict(MEM[t]["train"].astype(int), lr_frozen(mean_loss(t), DEV_L, P0, PIN0))
    assert abs(got - exp) < 6e-4, (t, got, exp)

rng = np.random.default_rng(20261017)                         # same dev bootstrap as earlier notebooks
pos_i, neg_i = np.flatnonzero(LABELS == 1), np.flatnonzero(LABELS == 0)
boot = [np.concatenate([rng.choice(pos_i, pos_i.size), rng.choice(neg_i, neg_i.size)]) for _ in range(2000)]
print(f"Frozen LR4 reproduced: AUC={roc_auc_score(LABELS, LR4_DEV):.4f} | strict TPR={tpr_strict(LABELS, LR4_DEV):.4f}; "
      "STAGED-5 mock values reproduced.")

Frozen LR4 reproduced: AUC=0.9744 | strict TPR=0.7120; STAGED-5 mock values reproduced.


In [5]:
# RC-0 — ReCaLL setup. After N-1..N-4 (GPU). Adapters read-only; writes only to cache/recall/.
import time, random
from datasets import load_dataset
RC = DRIVE / "cache" / "recall"; RC.mkdir(parents=True, exist_ok=True)
PREFIX_TOKENS, N_PREFIX, RC_BATCH, PREFIX_SEED = 128, 3, 8, 20261018
X_MAX = MAX_LENGTH - PREFIX_TOKENS                    # 384
txt = lambda r: r if isinstance(r, str) else r["text"]
FROZEN8 = ["inA", "inB", "p2A", "p2B", "p3A", "p3B", "p4A", "p4B"]
P56 = ["p5A", "p5B", "p6A", "p6B"]

def adapter_dir(name):
    for p in (DRIVE / "models" / name, DRIVE / name, ROOT / "models" / name):
        if (p / "adapter_model.safetensors").exists():
            return p
    raise FileNotFoundError(name)
def data_file(name):
    for p in (DRIVE / "data" / name, DRIVE / f"data_{name}", ROOT / "data" / name):
        if (p / "train_finetune.json").exists():
            return p / "train_finetune.json"
    raise FileNotFoundError(name)
RC_ADAPTERS = {"dev_target": ADAPTER_DIR} | {n: adapter_dir(f"shadow_{n}") for n in FROZEN8 + P56} | \
              {"shadow1001": adapter_dir("shadow_1001"), "shadow1002": adapter_dir("shadow_1002"),
               "val_target": ROOT / "models" / "val" / MODEL_NAME, "final_target": ROOT / "models" / "final" / MODEL_NAME}
ADSHA = {m: hashlib.sha256((Path(p) / "adapter_model.safetensors").read_bytes()).hexdigest() for m, p in RC_ADAPTERS.items()}

# Prefix candidates: WikiText-103 validation split; exclude eval texts and every known training set.
known_train = {norm(txt(r)) for r in read_json(ROOT / "data" / "train" / "train_finetune.json")}
for n in FROZEN8 + P56 + ["1001", "1002"]:
    known_train |= {norm(txt(r)) for r in read_json(data_file(f"shadow_{n}"))}
wv = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="validation")
cands = list(dict.fromkeys(" ".join(t.split()) for t in wv["text"] if isinstance(t, str) and t.strip()))
cands = [t for t in cands if not t.startswith("=")]
cl = [len(ids) for ids in tokenizer(cands, add_special_tokens=False)["input_ids"]]
cands = [t for t, l in zip(cands, cl) if l >= 25]
pk = prefix_keys(cands)
clean = [t for t, k in zip(cands, pk) if norm(t) not in EVAL_NORM_SET and k not in EVAL_PREF_SET and norm(t) not in known_train]
print(f"Validation-split paragraphs: {len(cands)} usable, {len(clean)} after exclusions")

rng = random.Random(PREFIX_SEED); order = clean[:]; rng.shuffle(order)
PREFIXES, PREFIX_SRC, pos = [], [], 0
for k in range(N_PREFIX):
    ids, used = [], []
    while len(ids) < PREFIX_TOKENS:
        used.append(order[pos]); pos += 1
        ids = tokenizer(" ".join(used), add_special_tokens=False)["input_ids"]
    PREFIXES.append(ids[:PREFIX_TOKENS]); PREFIX_SRC.append(used)
assert all(len(p) == PREFIX_TOKENS for p in PREFIXES)
PREFIX_SHA = hashlib.sha256(np.asarray(PREFIXES, dtype="<i4").tobytes()).hexdigest()
json.dump({"prefix_sha": PREFIX_SHA, "prefix_tokens": PREFIX_TOKENS, "n_prefix": N_PREFIX, "seed": PREFIX_SEED,
           "source": "WikiText-103 validation split (training draws from the train split)",
           "paragraphs": PREFIX_SRC, "prefix_ids": PREFIXES}, open(RC / "prefixes.json", "w"), indent=1)

X_IDS = [tokenizer(t, truncation=True, max_length=X_MAX)["input_ids"] for t in TEXTS["train"]]
FULL_IDS = tokenizer(TEXTS["train"], truncation=True, max_length=MAX_LENGTH)["input_ids"]
assert all(x == f[:len(x)] for x, f in zip(X_IDS, FULL_IDS))
print(f"Prefixes built ({PREFIX_SHA[:12]}); texts truncated to {X_MAX} tokens: "
      f"{sum(len(f) > X_MAX for f in FULL_IDS)} of 2000 affected")
for k, p in enumerate(PREFIXES):
    print(f"prefix {k}: {tokenizer.decode(p)[:120]!r} …")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Validation-split paragraphs: 1639 usable, 1638 after exclusions
Prefixes built (4e25c89a2079); texts truncated to 384 tokens: 24 of 2000 affected
prefix 0: 'The fourth installment in the Slammiversary name was announced in January 2008 to take place on June 8 . In March 2008 ,' …
prefix 1: 'The depiction of the police attempting to shut down the video shoot due to safety concerns actually happened during film' …
prefix 2: 'Roger Ebert gave the film three and a half stars out of four , calling it " well @-@ cast " and commending it for playin' …


In [6]:
# RC-1 — Unconditional and prefix-conditioned losses of x[1:], all 17 models, dev texts. fp16 eval, batch 8.
CACHE_DEV = {"dev_target": TGT["train"]} | {n: REF[n]["train"] for n in FROZEN8} | \
            {n: split_npz(OLD / f"train_shadow_{n}_losses.npz") for n in P56} | \
            {"shadow1001": split_npz(OLD / "train_shadow_1001_losses.npz", "shadow_losses"),
             "shadow1002": split_npz(OLD / "train_shadow_1002_losses.npz"),
             "val_target": split_npz(OLD / "train_ref_val_losses.npz"), "final_target": split_npz(OLD / "train_ref_final_losses.npz")}

def rc_expected(m):
    return {"model": m, "adapter_sha256": ADSHA[m], "dev_order_sha": DEV_ORDER_SHA, "prefix_sha": PREFIX_SHA,
            "prefix_tokens": PREFIX_TOKENS, "n_prefix": N_PREFIX, "x_max": X_MAX, "dtype": "float16", "batch": RC_BATCH,
            "scored": "x[1:] in both passes", "base_model": BASE_MODEL_NAME}
def rc_valid(m):
    pj, pz = RC / f"dev_{m}.json", RC / f"dev_{m}.npz"
    if not (pj.exists() and pz.exists()): return False
    meta = json.load(open(pj))
    return meta.get("status") == "done" and all(meta.get(k) == v for k, v in rc_expected(m).items())

@torch.inference_mode()
def rc_run(model, seqs, offset):
    res = []
    for s in range(0, len(seqs), RC_BATCH):
        ch = seqs[s:s + RC_BATCH]; L = max(map(len, ch))
        ids = torch.full((len(ch), L), tokenizer.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(ch), L), dtype=torch.long)
        for i, q in enumerate(ch):
            ids[i, :len(q)] = torch.tensor(q); att[i, :len(q)] = 1
        ids, att = ids.to(device), att.to(device)
        logits = model(input_ids=ids, attention_mask=att, use_cache=False).logits
        for i, q in enumerate(ch):
            n = len(q) - offset - 1
            res.append(F.cross_entropy(logits[i, offset:offset + n].float(), ids[i, offset + 1:offset + 1 + n],
                                       reduction="none").cpu().numpy())
        del logits
    return res

T_SC = np.array([len(x) - 1 for x in X_IDS])
for m, path in RC_ADAPTERS.items():
    if rc_valid(m):
        print(f"{m}: cached and verified"); continue
    t0 = time.perf_counter()
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=model_dtype)
    model = PeftModel.from_pretrained(base, path); model.config.pad_token_id = tokenizer.pad_token_id; model.to(device).eval()
    json.dump(rc_expected(m) | {"status": "running"}, open(RC / f"dev_{m}.json", "w"), indent=1)
    u = rc_run(model, X_IDS, 0)
    c = [rc_run(model, [P + x for x in X_IDS], PREFIX_TOKENS) for P in PREFIXES]
    assert all(a.size == t for a, t in zip(u, T_SC)) and all(a.size == t for ck in c for a, t in zip(ck, T_SC))
    u_mean = np.array([a.mean() for a in u], dtype=np.float64)
    c_mean = np.array([[a.mean() for a in ck] for ck in c], dtype=np.float64)
    assert np.isfinite(u_mean).all() and np.isfinite(c_mean).all()
    cache_diff = float(np.max(np.abs(u_mean - np.array([cd[:t].mean() for cd, t in zip(CACHE_DEV[m], T_SC)]))))
    np.savez(RC / f"dev_{m}.npz", dev_order_sha=np.array(DEV_ORDER_SHA), T=T_SC, u=u_mean, c=c_mean)
    json.dump(rc_expected(m) | {"status": "done", "minutes": (time.perf_counter() - t0) / 60,
                                "max_abs_u_vs_cached_loss": cache_diff}, open(RC / f"dev_{m}.json", "w"), indent=1)
    assert hashlib.sha256((Path(path) / "adapter_model.safetensors").read_bytes()).hexdigest() == ADSHA[m]
    model.to("cpu"); del model, base; gc.collect(); torch.cuda.empty_cache()
    d = (c_mean - u_mean).mean(0)
    print(f"{m}: {(time.perf_counter() - t0) / 60:.1f} min | |u − cached| max {cache_diff:.1e} | "
          f"mean d = {d.mean():+.3f} (sd {d.std():.3f})")
print("Extraction complete.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

dev_target: 0.3 min | |u − cached| max 6.6e-03 | mean d = +0.262 (sd 0.178)
inA: 0.2 min | |u − cached| max 8.8e-03 | mean d = +0.256 (sd 0.173)
inB: 0.2 min | |u − cached| max 7.0e-03 | mean d = +0.251 (sd 0.174)
p2A: 0.2 min | |u − cached| max 7.6e-03 | mean d = +0.254 (sd 0.175)
p2B: 0.2 min | |u − cached| max 8.7e-03 | mean d = +0.263 (sd 0.177)
p3A: 0.2 min | |u − cached| max 7.4e-03 | mean d = +0.259 (sd 0.175)
p3B: 0.2 min | |u − cached| max 8.2e-03 | mean d = +0.250 (sd 0.174)
p4A: 0.2 min | |u − cached| max 7.5e-03 | mean d = +0.258 (sd 0.175)
p4B: 0.2 min | |u − cached| max 7.7e-03 | mean d = +0.260 (sd 0.178)
p5A: 0.2 min | |u − cached| max 8.1e-03 | mean d = +0.260 (sd 0.174)
p5B: 0.2 min | |u − cached| max 8.4e-03 | mean d = +0.254 (sd 0.175)
p6A: 0.2 min | |u − cached| max 9.7e-03 | mean d = +0.249 (sd 0.171)
p6B: 0.2 min | |u − cached| max 7.1e-03 | mean d = +0.264 (sd 0.178)
shadow1001: 0.2 min | |u − cached| max 9.2e-03 | mean d = +0.243 (sd 0.169)
shadow1002: 0.2 min 

In [7]:
# RC-3 — ReCaLL u/c on val and final texts, same prefixes, for every model in LR4's val/final reference sets.
PH_MODELS = ["dev_target", "val_target", "final_target", "shadow1001", "shadow1002"] + FROZEN8
X_PH = {ph: [tokenizer(t, truncation=True, max_length=X_MAX)["input_ids"] for t in TEXTS[ph]] for ph in ("val", "final")}
ORDER_SHA = {ph: hashlib.sha256("\x1e".join(TEXTS[ph]).encode()).hexdigest() for ph in ("val", "final")}
def ph_expected(ph, m):
    return rc_expected(m) | {"phase": ph, "dev_order_sha": None, "order_sha": ORDER_SHA[ph]}
def ph_valid(ph, m):
    pj, pz = RC / f"{ph}_{m}.json", RC / f"{ph}_{m}.npz"
    if not (pj.exists() and pz.exists()): return False
    meta = json.load(open(pj))
    return meta.get("status") == "done" and all(meta.get(k) == v for k, v in ph_expected(ph, m).items())

for m in PH_MODELS:
    todo = [ph for ph in ("val", "final") if not ph_valid(ph, m)]
    if not todo:
        print(f"{m}: cached"); continue
    t0 = time.perf_counter()
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=model_dtype)
    model = PeftModel.from_pretrained(base, RC_ADAPTERS[m]); model.config.pad_token_id = tokenizer.pad_token_id; model.to(device).eval()
    for ph in todo:
        T_ph = np.array([len(x) - 1 for x in X_PH[ph]])
        u = rc_run(model, X_PH[ph], 0)
        c = [rc_run(model, [P + x for x in X_PH[ph]], PREFIX_TOKENS) for P in PREFIXES]
        u_mean = np.array([a.mean() for a in u]); c_mean = np.array([[a.mean() for a in ck] for ck in c])
        assert np.isfinite(u_mean).all() and np.isfinite(c_mean).all() and all(a.size == t for a, t in zip(u, T_ph))
        np.savez(RC / f"{ph}_{m}.npz", order_sha=np.array(ORDER_SHA[ph]), T=T_ph, u=u_mean, c=c_mean)
        json.dump(ph_expected(ph, m) | {"status": "done"}, open(RC / f"{ph}_{m}.json", "w"), indent=1)
    assert hashlib.sha256((Path(RC_ADAPTERS[m]) / "adapter_model.safetensors").read_bytes()).hexdigest() == ADSHA[m]
    model.to("cpu"); del model, base; gc.collect(); torch.cuda.empty_cache()
    print(f"{m}: {todo} done in {(time.perf_counter() - t0) / 60:.1f} min")
print("Val/final ReCaLL extraction complete.")

dev_target: ['val', 'final'] done in 0.4 min
val_target: ['val', 'final'] done in 0.4 min
final_target: ['val', 'final'] done in 0.4 min
shadow1001: ['val', 'final'] done in 0.4 min
shadow1002: ['val', 'final'] done in 0.4 min
inA: ['val', 'final'] done in 0.4 min
inB: ['val', 'final'] done in 0.4 min
p2A: ['val', 'final'] done in 0.4 min
p2B: ['val', 'final'] done in 0.4 min
p3A: ['val', 'final'] done in 0.4 min
p3B: ['val', 'final'] done in 0.4 min
p4A: ['val', 'final'] done in 0.4 min
p4B: ['val', 'final'] done in 0.4 min
Val/final ReCaLL extraction complete.


In [8]:
# QR-0 — Non-member pool (WikiText-103 validation + test splits) scored by all 17 models, identical to the cache setup.
QR = DRIVE / "cache" / "qr_pool"; QR.mkdir(parents=True, exist_ok=True)
pool = []
for split in ("validation", "test"):
    ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split=split)
    pool += [" ".join(t.split()) for t in ds["text"] if isinstance(t, str) and t.strip()]
pool = [t for t in dict.fromkeys(pool) if not t.startswith("=")]
pl = [len(ids) for ids in tokenizer(pool, add_special_tokens=False)["input_ids"]]
pool = [t for t, l in zip(pool, pl) if l >= 25]
pk = prefix_keys(pool)
used_prefix = {norm(p) for src in PREFIX_SRC for p in src}
POOL = [t for t, k in zip(pool, pk) if norm(t) not in EVAL_NORM_SET and k not in EVAL_PREF_SET
        and norm(t) not in known_train and norm(t) not in used_prefix]
POOL_SHA = hashlib.sha256("\x1e".join(POOL).encode()).hexdigest()
json.dump({"source": "WikiText-103 validation+test (all training uses the train split)", "n": len(POOL),
           "sha": POOL_SHA, "texts": POOL}, open(QR / "pool.json", "w"))
print(f"Pool: {len(POOL)} non-member paragraphs ({POOL_SHA[:12]})")

@torch.inference_mode()
def collect_losses(model, texts, batch_size=2):
    """Identical to the eval-text caches: fp16 forward, CE on fp32 logits, batch 2, right padding, max_length 512."""
    out = []
    for s in range(0, len(texts), batch_size):
        b = tokenizer(texts[s:s + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        b = {k: v.to(device) for k, v in b.items()}
        tgt, mask = b["input_ids"][:, 1:], b["attention_mask"][:, 1:].bool()
        logits = model(**b, use_cache=False).logits[:, :-1, :]
        loss = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), tgt.reshape(-1), reduction="none").reshape(tgt.shape)
        out += [row[m].float().cpu().numpy() for row, m in zip(loss, mask)]
    return out

for m, path in RC_ADAPTERS.items():
    pz, pj = QR / f"pool_{m}.npz", QR / f"pool_{m}.json"
    if pz.exists() and pj.exists() and json.load(open(pj)).get("pool_sha") == POOL_SHA and \
       json.load(open(pj)).get("adapter_sha256") == ADSHA[m]:
        print(f"{m}: cached"); continue
    t0 = time.perf_counter()
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=model_dtype)
    model = PeftModel.from_pretrained(base, path); model.config.pad_token_id = tokenizer.pad_token_id; model.to(device).eval()
    ls = collect_losses(model, POOL)
    assert len(ls) == len(POOL) and all(np.isfinite(a).all() for a in ls)
    np.savez_compressed(pz, lengths=np.array([a.size for a in ls]), losses=np.concatenate(ls))
    json.dump({"model": m, "adapter_sha256": ADSHA[m], "pool_sha": POOL_SHA, "dtype": "float16", "batch": 2},
              open(pj, "w"), indent=1)
    model.to("cpu"); del model, base; gc.collect(); torch.cuda.empty_cache()
    print(f"{m}: {(time.perf_counter() - t0) / 60:.1f} min")
print("Pool scoring complete.")

Pool: 3462 non-member paragraphs (00d71718f976)
dev_target: 0.2 min
inA: 0.2 min
inB: 0.2 min
p2A: 0.2 min
p2B: 0.2 min
p3A: 0.2 min
p3B: 0.2 min
p4A: 0.2 min
p4B: 0.2 min
p5A: 0.2 min
p5B: 0.2 min
p6A: 0.2 min
p6B: 0.2 min
shadow1001: 0.2 min
shadow1002: 0.2 min
val_target: 0.2 min
final_target: 0.2 min
Pool scoring complete.


In [9]:
# QR-1 — Gate: residual of each target vs its LR4-style reference set on known non-members. No labels used.
from numpy.linalg import lstsq
PM = {}
for m in RC_ADAPTERS:
    z = np.load(QR / f"pool_{m}.npz"); a = np.split(z["losses"], np.cumsum(z["lengths"])[:-1])
    PM[m] = np.array([x.mean() for x in a])
PT = np.load(QR / f"pool_dev_target.npz")["lengths"].astype(float); SQP = np.sqrt(PT)
PAIRED8 = FROZEN8
REFSETS = {   # LR4's own reference rule for each real target; mocks: other pairs + the same fixed models as dev
    "dev_target (real)":   PAIRED8 + ["shadow1001", "shadow1002", "val_target", "final_target"],
    "val_target (real)":   PAIRED8 + ["shadow1001", "shadow1002", "final_target", "dev_target"],
    "final_target (real)": PAIRED8 + ["shadow1001", "shadow1002", "dev_target", "val_target"]}
PAIR_OF = {n: (n[:-1] + "A", n[:-1] + "B") if n[:2] in ("p2", "p3", "p4", "p5", "p6") else ("inA", "inB") for n in FROZEN8 + P56}
for t in ["p4A", "p4B", "p5A", "p5B", "p6A", "p6B", "inA", "p2A", "p3B"]:
    REFSETS[f"{t} (shadow)"] = [n for n in PAIRED8 if n not in PAIR_OF[t]] + ["shadow1001", "shadow1002", "val_target", "final_target"]

rows = []
for label, refs in REFSETS.items():
    t = label.split(" ")[0]
    R = np.stack([PM[n] for n in refs]) * SQP; u = PM[t] * SQP
    mu, sd = R.mean(0), R.std(0, ddof=1)
    r = u - mu                                                     # target's non-member residual (√T scale)
    z = r / np.sqrt(sd ** 2 * (1 + 1 / len(refs)))
    X = np.column_stack([np.ones_like(mu), mu, SQP, sd])
    beta = lstsq(X, r, rcond=None)[0]; r2 = 1 - ((r - X @ beta) ** 2).mean() / r.var()
    loo = np.concatenate([(R[i] - np.delete(R, i, 0).mean(0)) for i in range(len(refs))])   # refs vs each other
    rows.append({"target": label, "n refs": len(refs), "mean residual": r.mean(), "sd residual": r.std(),
                 "sd ref-vs-ref (LOO)": loo.std(), "ratio sd": r.std() / loo.std(), "mean z": z.mean(), "sd z": z.std(),
                 "z > 2.33 (frac)": float((z < -2.33).mean()), "R² of residual on (μ, √T, sd)": r2})
gate = pd.DataFrame(rows)
pd.set_option("display.width", 240)
print(f"Pool of {len(PT)} known non-members; residual = target √T·mean-loss − mean of its references:")
print(gate.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
gate.to_csv(QR / "gate.csv", index=False)

Pool of 3462 known non-members; residual = target √T·mean-loss − mean of its references:
             target  n refs  mean residual  sd residual  sd ref-vs-ref (LOO)  ratio sd  mean z  sd z  z > 2.33 (frac)  R² of residual on (μ, √T, sd)
  dev_target (real)      12         -0.001        0.219                0.208     1.056   0.005 1.177            0.023                          0.002
  val_target (real)      12         -0.012        0.231                0.207     1.118  -0.080 1.239            0.040                          0.007
final_target (real)      12          0.001        0.239                0.206     1.160   0.028 1.251            0.032                          0.009
       p4A (shadow)      10         -0.004        0.215                0.208     1.035  -0.035 1.220            0.031                          0.005
       p4B (shadow)      10         -0.002        0.218                0.208     1.048  -0.012 1.222            0.027                          0.000
       p5A (shado

In [10]:
# QR-2 — LR4 with a per-target, label-free variance inflation c_t estimated on the non-member pool. Dev + fresh mocks.
def lr_infl(mt, L, P, Pin, c=1.0, kappa=8):
    sq = np.sqrt(L); U, u = P * sq, mt * sq
    fin = np.isfinite(P); m_in, m_out = Pin & fin, ~Pin & fin
    n_in, n_out = m_in.sum(0), m_out.sum(0); Uz = np.where(fin, U, 0.0)
    mu_in = np.where(m_in, Uz, 0).sum(0) / n_in; mu_out = np.where(m_out, Uz, 0).sum(0) / n_out
    rss_in = np.where(m_in, (Uz - mu_in) ** 2, 0).sum(0); rss_out = np.where(m_out, (Uz - mu_out) ** 2, 0).sum(0)
    df_in, df_out = np.maximum(n_in - 1, 0), np.maximum(n_out - 1, 0)
    pool_out = rss_out.sum() / df_out.sum(); pool_in = rss_in.sum() / df_in.sum() if df_in.sum() > 0 else pool_out
    s_in = c * (rss_in + kappa * pool_in) / (df_in + kappa) * (1 + 1 / n_in)
    s_out = c * (rss_out + kappa * pool_out) / (df_out + kappa) * (1 + 1 / n_out)
    return (-(u - mu_in) ** 2 / (2 * s_in) - 0.5 * np.log(s_in)) - (-(u - mu_out) ** 2 / (2 * s_out) - 0.5 * np.log(s_out))

def pool_c(t, refs):
    """Excess spread of target t around its refs on known non-members, relative to LR's exchangeable predictive variance."""
    R = np.stack([PM[n] for n in refs]) * SQP; r = PM[t] * SQP - R.mean(0)
    return float(np.mean(r ** 2) / np.mean(R.var(0, ddof=1) * (1 + 1 / len(refs))))

# Dev-text statistics for all models (cached losses), memberships for p5/p6.
DEVM = {n: np.array([a.mean() for a in CACHE_DEV[n]]) for n in CACHE_DEV}
for n in P56:
    MEM[n] = membership([txt(r) for r in read_json(data_file(f"shadow_{n}"))])
FIXN = ["shadow1001", "shadow1002", "val_target", "final_target"]
def lr_target(t, refs, c):
    P = np.stack([DEVM[n] for n in refs])
    Pin = np.stack([MEM[n]["train"] if n in MEM else np.zeros(2000, bool) for n in refs])
    return lr_infl(DEVM[t], DEV_L, P, Pin, c)
assert np.allclose(lr_target("dev_target", FROZEN8 + FIXN, 1.0), LR4_DEV, rtol=0, atol=1e-6)

def boot_for(y, seed):
    rng = np.random.default_rng(seed); pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    return [np.concatenate([rng.choice(pos, pos.size), rng.choice(neg, neg.size)]) for _ in range(2000)]

rows = []
cases = [("dev_target", FROZEN8 + FIXN, LABELS, boot)]
for t in ["p4A", "p4B", "p5A", "p5B", "p6A", "p6B"]:
    own = (t[:-1] + "A", t[:-1] + "B")
    cases.append((t, [n for n in FROZEN8 if n not in own] + FIXN, MEM[t]["train"].astype(int), None))
for t, refs, y, bix in cases:
    c = pool_c(t, refs)
    base, infl = lr_target(t, refs, 1.0), lr_target(t, refs, c)
    bix = bix or boot_for(y, 300 + len(rows))
    d = np.array([strict_fast(infl[ix], y[ix]) - strict_fast(base[ix], y[ix]) for ix in bix]); lo, hi = np.percentile(d, [2.5, 97.5])
    rows.append({"target": t + (" (real)" if t == "dev_target" else " (fresh mock)"), "refs": len(refs), "c (pool)": c,
                 "LR TPR": tpr_strict(y, base), "inflated TPR": tpr_strict(y, infl),
                 "Δ": tpr_strict(y, infl) - tpr_strict(y, base), "CI": f"[{lo:+.3f}, {hi:+.3f}]",
                 "LR AUC": roc_auc_score(y, base), "inflated AUC": roc_auc_score(y, infl)})
res = pd.DataFrame(rows)
print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
mock = res[res.target.str.contains("mock")]
PASS = bool(mock["Δ"].mean() > 0 and (mock["Δ"] > 0).sum() >= 4 and res.iloc[0]["Δ"] >= 0)
print(f"\nMean Δ over 6 fresh mocks: {mock['Δ'].mean():+.4f} ({int((mock['Δ'] > 0).sum())}/6 positive) | "
      f"dev Δ {res.iloc[0]['Δ']:+.4f} → {'PASS' if PASS else 'FAIL'}")
print("c for validation/final targets:", {t: round(pool_c(t, REFSETS[f'{t} (real)']), 3) for t in ("val_target", "final_target")})
res.to_csv(QR / "variance_inflation_eval.csv", index=False)

           target  refs  c (pool)  LR TPR  inflated TPR       Δ               CI  LR AUC  inflated AUC
dev_target (real)    12    1.1219  0.7120        0.7130  0.0010 [-0.005, +0.008]  0.9744        0.9745
 p4A (fresh mock)    10    1.0823  0.7908        0.7928  0.0020 [-0.004, +0.005]  0.9798        0.9799
 p4B (fresh mock)    10    1.1088  0.6908        0.6928  0.0020 [-0.005, +0.006]  0.9729        0.9729
 p5A (fresh mock)    12    1.1363  0.7467        0.7457 -0.0010 [-0.003, +0.005]  0.9794        0.9796
 p5B (fresh mock)    12    1.0953  0.7287        0.7228 -0.0059 [-0.008, +0.006]  0.9778        0.9778
 p6A (fresh mock)    12    1.1125  0.8008        0.8018  0.0010 [-0.004, +0.005]  0.9814        0.9814
 p6B (fresh mock)    12    1.0907  0.6815        0.6826  0.0010 [-0.002, +0.006]  0.9718        0.9718

Mean Δ over 6 fresh mocks: -0.0001 (4/6 positive) | dev Δ +0.0010 → FAIL
c for validation/final targets: {'val_target': 1.262, 'final_target': 1.354}
